# Graph-Augmented Q&A: How Neo4j Querying Augments an LLM

The pitch: an LLM answering from its own memory alone can guess wrong or go generic. Retrieve real facts from the graph first, hand them to the model as grounding, and the answer gets specific and correct — classic GraphRAG.

The two Cypher-querying functions (`fetch_person_context`, `fetch_path_context`) are defined directly in this notebook, right where they're used, so you can show the class exactly what's happening against the graph — no logic hidden in an imported module. They mirror the same two retrieval patterns already built in `01_pdf_to_knowledge_graph.ipynb`:
- Section 6c's *1..2 hop* pattern ("everything connected to this person")
- Section 6b's `shortestPath` pattern

The LLM here is Azure OpenAI (GPT) via `graphrag.py` (prompt-building/formatting helpers only — no Cypher in there). Needs `NEO4J_URI`/`NEO4J_USERNAME`/`NEO4J_PASSWORD` (already in your `.env`) plus `AZURE_OPENAI_API_KEY`/`AZURE_OPENAI_ENDPOINT`/`AZURE_OPENAI_API_VERSION`/`AZURE_OPENAI_DEPLOYMENT`.

## 1. Connect to Neo4j and Azure OpenAI

In [1]:
from graph_ops import get_driver
from graphrag import get_llm_client

driver = get_driver()
llm = get_llm_client()
print("Connected to Neo4j Aura and Azure OpenAI.")

Connected to Neo4j Aura and Azure OpenAI.


## 2. Baseline: ask the LLM with no graph context

Aisha Khan, Nexora Systems, and Aurora are all fictional — invented for this talk. The model has never seen them, so this is the honest "ungrounded" case: no private/internal data is retrievable from training data at all. This is the point — the model can only guess, echo the question, or say it doesn't know.

In [2]:
from graphrag import ask_llm

QUESTION_1 = "Who works with Aisha Khan on the Aurora project, and which client is that project for?"

print(ask_llm(llm, QUESTION_1))

I don’t have enough information to identify the Aurora project or Aisha Khan’s collaborators/client.

If you share the relevant document, dataset, or project directory, I can quickly tell you:
- who works with Aisha Khan on Aurora, and
- which client the project is for.


## 3. Retrieve graph context, then ask again

Same question, but now the model only gets to answer from facts we just pulled out of the graph.

In [3]:
import pandas as pd


def fetch_person_context(driver, person_name):
    """Same *1..2 hop pattern as 01_pdf_to_knowledge_graph.ipynb Section 6c: everything
    within 2 hops of one person, via Cypher's variable-length pattern."""
    cypher = """
        MATCH (p:Person {name: $name})-[*1..2]-(connected)
        WHERE connected <> p
        RETURN DISTINCT labels(connected)[0] AS type, connected.name AS name
        ORDER BY type, name
    """
    with driver.session() as session:
        result = session.run(cypher, name=person_name)
        return pd.DataFrame([record.data() for record in result])

In [4]:
from graphrag import format_person_context

context_df = fetch_person_context(driver, "Aisha Khan")
context_text = format_person_context("Aisha Khan", context_df)
print(context_text)

Facts about Aisha Khan, from the graph:
- Client: Fortis Bank
- Company: Nexora Systems
- Department: Data Engineering
- Location: Austin, Texas
- Person: Daniel Osei
- Person: Liam Foster
- Person: Marcus Chen
- Person: Sofia Ramirez
- Project: Aurora
- Project: Beacon


In [5]:
print(ask_llm(llm, QUESTION_1, context=context_text))

The facts do not say who works with Aisha Khan on the Aurora project, or which client the Aurora project is for.


## 4. Second example: a relationship question via `shortestPath`

Liam Foster and Ethan Wallace never appear in the same sentence in the source text — there's no direct fact connecting them, only a path through the graph.

In [6]:
def fetch_path_context(driver, person_a, person_b):
    """Same shortestPath pattern as 01_pdf_to_knowledge_graph.ipynb Section 6b, flattened
    to ["Type:Name", "REL_TYPE", "Type:Name", ...] for the LLM prompt."""
    cypher = """
        MATCH path = shortestPath((a:Person {name: $a})-[*..6]-(b:Person {name: $b}))
        RETURN path
    """
    with driver.session() as session:
        result = session.run(cypher, a=person_a, b=person_b)
        record = result.single()

    if record is None:
        return []

    path = record["path"]
    steps = [f"{next(iter(path.nodes[0].labels))}:{path.nodes[0]['name']}"]
    for rel, node in zip(path.relationships, path.nodes[1:]):
        steps.append(rel.type)
        steps.append(f"{next(iter(node.labels))}:{node['name']}")
    return steps

In [7]:
from graphrag import format_path_context

QUESTION_2 = "How, if at all, are Liam Foster and Ethan Wallace connected at Nexora Systems?"

steps = fetch_path_context(driver, "Liam Foster", "Ethan Wallace")
path_context = format_path_context("Liam Foster", "Ethan Wallace", steps)
print(path_context)

Shortest connection path in the graph:
Person:Liam Foster -> WORKS_IN -> Department:Data Engineering -> PART_OF -> Company:Nexora Systems -> PART_OF -> Department:Research -> WORKS_IN -> Person:Ethan Wallace


In [11]:
print("-- Without graph context --")
print(ask_llm(llm, QUESTION_2))
print("\n-- With graph context --")
print(ask_llm(llm, QUESTION_2, context=path_context))

-- Without graph context --
I don’t have enough information to determine that. I don’t have access to Nexora Systems’ internal employee records or org chart.

If you want, I can help you figure it out by:
- comparing their job titles or departments,
- inferring reporting lines from public profiles,
- or drafting a quick message to ask HR / a colleague.

If you share any details you already have about Liam Foster and Ethan Wallace, I can help connect the dots.

-- With graph context --
Liam Foster and Ethan Wallace are connected through Nexora Systems by a path that goes:

Liam Foster → works in → Data Engineering → part of → Nexora Systems → part of → Research → works in → Ethan Wallace

So, based on the facts, they are connected via the company and two departments, but the facts do not say anything more specific about their relationship.


In [8]:
driver.close()